# Practicum 1: opfrisser, een model trainen en de fouten lezen

AI & Ethics (A-ET), blok 0. Dit notebook doet dezelfde stappen als de Orange-versie in kennispakket 0. De vragen en de evidence card zijn hetzelfde.

Je hebt `suribank_kredietaanvragen.csv` nodig, in dezelfde map als dit notebook.

**Let op:** Python rekent op een paar punten anders dan Orange, bijvoorbeeld bij het splitsen van de data en het bouwen van de boom. Je getallen kunnen daardoor iets afwijken van die van medestudenten met Orange. De vragen gaan over wat je ziet, niet over het precieze getal.

## Stap 1. Data inladen en het label instellen

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.dummy import DummyClassifier

DOEL = "terugbetaald"   # het label: betaalde de aanvrager terug?
ID = "aanvraag_id"      # een nummer zegt niets over de aanvrager, dus dit gebruiken we niet

def voorbewerking(kenmerken, data, schalen):
    tekst = [k for k in kenmerken if data[k].dtype == object]
    getal = [k for k in kenmerken if k not in tekst]
    stappen = [SimpleImputer(strategy="median")] + ([StandardScaler()] if schalen else [])
    return ColumnTransformer([
        ("tekst", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), tekst),
        ("getal", make_pipeline(*stappen), getal)], verbose_feature_names_out=False)

def logistische_regressie(kenmerken, data):
    return Pipeline([("prep", voorbewerking(kenmerken, data, schalen=True)), ("model", LogisticRegression(max_iter=1000))])

def boom(kenmerken, data, diepte=None):
    return Pipeline([("prep", voorbewerking(kenmerken, data, schalen=False)),
                     ("model", DecisionTreeClassifier(max_depth=diepte, min_samples_leaf=2, min_samples_split=5, random_state=0))])

def kans_ja(model, data, kenmerken):
    ja = list(model.classes_).index("ja")
    return model.predict_proba(data[kenmerken])[:, ja]

data = pd.read_csv("suribank_kredietaanvragen.csv")
kenmerken = [k for k in data.columns if k not in (DOEL, ID)]
print(len(data), "aanvragen")
print("Kenmerken:", kenmerken)
data.head()

## Stap 2. Terugbetalen per groep

Welk deel betaalde terug, per regio? En hoe is het soort werk per regio verdeeld?

In [ ]:
terug = pd.crosstab(data["regio"], data[DOEL], normalize="index").round(3) * 100
print("Terugbetaald per regio (procent):")
print(terug)
print()
werk = pd.crosstab(data["regio"], data["dienstverband"], normalize="index").round(3) * 100
print("Soort werk per regio (procent):")
print(werk)

## Stap 3. Data splitsen

70% om te leren, 30% om te testen. `stratify` zorgt dat in beide delen ongeveer evenveel terugbetalers zitten, net als in Orange. `random_state=7` legt vast welke aanvragers in welk deel komen, zodat je bij opnieuw draaien dezelfde uitkomst krijgt.

In [ ]:
from sklearn.model_selection import train_test_split
leer, test = train_test_split(data, train_size=0.7, stratify=data[DOEL], random_state=7)
print(len(leer), "om te leren,", len(test), "om te testen, waarvan", (test[DOEL] == "ja").sum(), "terugbetaald")

## Stap 4. Drie modellen

Logistische regressie, een beslisboom, en Constant: een model dat altijd het antwoord geeft dat het vaakst voorkomt. Dat is je ondergrens.

In [ ]:
modellen = {
    "Logistic Regression": logistische_regressie(kenmerken, data),
    "Tree": boom(kenmerken, data),
    "Constant": Pipeline([("prep", voorbewerking(kenmerken, data, schalen=False)), ("model", DummyClassifier(strategy="most_frequent"))]),
}
for naam, m in modellen.items():
    m.fit(leer[kenmerken], leer[DOEL])
print("Drie modellen geleerd")

## Stap 5 en 6. Testen en de verwarringsmatrix

CA is het deel dat het model goed voorspelt. De vier vakken laten zien welke fouten het model maakt. Precisie: van alle goedgekeurde aanvragen, welk deel betaalde terug? Recall: van alle goede betalers, welk deel werd goedgekeurd?

In [ ]:
rijen = []
for naam, m in modellen.items():
    voorspeld = m.predict(test[kenmerken])
    echt = test[DOEL].values
    tg = int(((voorspeld == "ja") & (echt == "ja")).sum())   # terecht goedgekeurd
    og = int(((voorspeld == "ja") & (echt == "nee")).sum())  # onterecht goedgekeurd
    oa = int(((voorspeld == "nee") & (echt == "ja")).sum())  # onterecht afgewezen
    ta = int(((voorspeld == "nee") & (echt == "nee")).sum()) # terecht afgewezen
    rijen.append({"model": naam, "CA": round((tg + ta) / len(echt) * 100, 1),
                  "terecht goedgekeurd": tg, "onterecht goedgekeurd": og,
                  "onterecht afgewezen": oa, "terecht afgewezen": ta,
                  "precisie ja": round(tg / (tg + og) * 100, 1) if tg + og else None,
                  "recall ja": round(tg / (tg + oa) * 100, 1)})
pd.DataFrame(rijen).set_index("model")

## Stap 7. Twijfelgevallen

De kans op *ja* per testaanvraag, volgens logistische regressie. We tonen de aanvragen met een kans dicht bij 50%.

In [ ]:
lr = modellen["Logistic Regression"]
t = test.copy()
t["kans op ja"] = kans_ja(lr, t, kenmerken).round(3)
twijfel = t.iloc[(t["kans op ja"] - 0.5).abs().argsort()[:8]]
twijfel[[ID, "regio", "dienstverband", "maandinkomen_srd", "gevraagd_bedrag_srd", "looptijd_maanden", "kans op ja", DOEL]]

## Extra bij vraag 6. District en regio weglaten

Wat gebeurt er als het model district en regio niet meer krijgt? Kijk naar de CA, en denk aan je antwoord op vraag 1. In practicum 2 (blok 1) meet je de fouten per regio.

In [ ]:
zonder = [k for k in kenmerken if k not in ("district", "regio")]
lr2 = logistische_regressie(zonder, data).fit(leer[zonder], leer[DOEL])
print("CA met district en regio:   ", round((lr.predict(test[kenmerken]) == test[DOEL]).mean() * 100, 1), "%")
print("CA zonder district en regio:", round((lr2.predict(test[zonder]) == test[DOEL]).mean() * 100, 1), "%")

## Extra. Hoe stabiel is dit?

De uitkomst hangt af van welke aanvragers toevallig in de testset komen. Hieronder splitsen we tien keer anders en kijken we naar het gemiddelde en de spreiding. Wat betekent dit voor een organisatie die zegt: "ons model haalt 70%"?

In [ ]:
uitkomsten = []
for seed in range(10):
    l, t = train_test_split(data, train_size=0.7, stratify=data[DOEL], random_state=seed)
    for naam, maak in [("Logistic Regression", lambda: logistische_regressie(kenmerken, data)), ("Tree", lambda: boom(kenmerken, data))]:
        m = maak().fit(l[kenmerken], l[DOEL])
        uitkomsten.append({"model": naam, "CA": (m.predict(t[kenmerken]) == t[DOEL]).mean() * 100})
pd.DataFrame(uitkomsten).groupby("model")["CA"].agg(["mean", "min", "max"]).round(1)

## Stap 8. Opslaan en vragen

Sla het notebook op. Beantwoord de zeven vragen uit kennispakket 0 en vul evidence card 1 in. Lever een schermafdruk van je notebook in.

1. Welk deel van de aanvragers betaalde terug aan de kust, en welk deel in het binnenland? Hoe verschilt het soort werk tussen de twee regio's?
2. Welk deel van de testaanvragen voorspelt elk model goed (CA)? Hoeveel procentpunt zijn Tree en Logistic Regression beter dan Constant?
3. Vul voor Logistic Regression de vier vakken in.
4. Welk model wijst meer goede betalers onterecht af? Welk model zou de bank kiezen, en welk model zouden aanvragers kiezen? Gebruik precisie en recall.
5. Zoek een aanvraag met een kans rond 50%. Wat zou jij als bank met deze aanvraag doen, en wie zou erover moeten beslissen?
6. Stel dat de bank district en regio weglaat. Is het risico op ongelijke behandeling van mensen uit het binnenland dan weg?
7. Denk aan jullie eigen oplossing uit A-AI1 en A-AI2. Welke fout kan die maken, en wie merkt de gevolgen?